# SoundMatch — Análisis de Concordancia Humano-Modelo (HMA)

Notebook de análisis de los resultados del experimento de juicio perceptual de similitud musical para la tesis del **Posgrado en Ciencia e Ingeniería de la Computación, UNAM**.

## Objetivo
Evaluar qué tan bien cuatro modelos de representación de audio aprendidos con *deep learning* (MusiCNN, VGG, Whisper Contrastivo MultiLabel y MusiCNN MultiSeñal) capturan la similitud musical percibida por humanos, mediante el cálculo de la métrica **Human-Model Agreement (HMA)**.

## Diseño experimental
Forced-choice de triplete: dado un audio de referencia (*anchor*) y dos opciones (A y B), el respondiente elige cuál de las dos suena más parecida a la referencia. Por cada triplete se calcula el voto mayoritario humano y se compara contra la decisión que hace cada modelo según su distancia coseno entre embeddings.

**HMA** = porcentaje de tripletes donde la elección del modelo coincide con el voto mayoritario humano.

## Estructura
1. Carga de datos y descripción del corpus
2. Estadística descriptiva de respondientes y respuestas
3. Análisis de votación por triplete
4. Concordancia entre expertos y público general
5. Cómputo de HMA por modelo
6. Comparación pareada entre modelos (test de McNemar)
7. Exportación de figuras y tablas para la tesis

## 0. Configuración

In [ ]:
import io
import os
import sqlite3
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics.pairwise import cosine_distances
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

# --- Paths: private database locations must be supplied explicitly ---
REPORTS_DIR = Path(os.environ.get(
    'DEEP_AUDIO_REPORTS_DIR', Path.cwd() / 'backend' / 'reports'
)).expanduser().resolve()

def required_private_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f'Set {variable} to an authorized private database snapshot')
    path = Path(value).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    return path

EVAL_DB  = required_private_path('HMA_EVAL_DB')
EMBED_DB = required_private_path('HMA_EMBEDDINGS_DB')
FIG_DIR  = REPORTS_DIR / 'figures'
TBL_DIR  = REPORTS_DIR / 'tables'
FIG_DIR.mkdir(exist_ok=True)
TBL_DIR.mkdir(exist_ok=True)

# --- Modelos a evaluar ---
MODELS = [
    ('musicnn',                          'msd',  'MusiCNN (MSD)'),
    ('vgg',                              'msd',  'VGG (MSD)'),
    ('whisper_contrastive_multilabel',   'base', 'Whisper Contrastivo (MultiLabel)'),
    ('musicnn_multisignal',              'msd',  'MusiCNN MultiSeñal'),
]
MODEL_NAMES = {m: name for m, _, name in MODELS}

# --- Estilo ---
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['font.family'] = 'DejaVu Sans'

print(f'Eval DB:   {EVAL_DB}  ({EVAL_DB.exists()})')
print(f'Embed DB:  {EMBED_DB} ({EMBED_DB.exists()})')
print(f'Figures →  {FIG_DIR}')
print(f'Tables  →  {TBL_DIR}')

## 1. Carga de datos

In [ ]:
with sqlite3.connect(EVAL_DB) as conn:
    triplets  = pd.read_sql('SELECT * FROM eval_triplets',  conn)
    responses = pd.read_sql(
        'SELECT id, session_id, triplet_id, choice, response_time_ms, created_at, respondent_type '
        'FROM eval_responses',
        conn,
    )

responses['created_at'] = pd.to_datetime(responses['created_at'])
responses['respondent_type'] = responses['respondent_type'].fillna('public').replace('', 'public')

print(f'Triplets cargados:   {len(triplets)}')
print(f'Respuestas cargadas: {len(responses)}')
print(f'Sesiones únicas:     {responses["session_id"].nunique()}')
responses.head()

In [ ]:
triplets.head()

## 2. Estadística descriptiva

### 2.1 Sesiones y respuestas por tipo de respondiente

In [ ]:
summary = (
    responses
    .groupby('respondent_type')
    .agg(
        sesiones_unicas=('session_id', 'nunique'),
        respuestas_totales=('id', 'count'),
        respuestas_promedio=('id', lambda s: s.count() / responses.loc[s.index, 'session_id'].nunique()),
    )
    .round(2)
)

# Sesiones completas (≥10 rondas)
session_counts = responses.groupby(['respondent_type', 'session_id']).size().reset_index(name='n')
completed = session_counts[session_counts['n'] >= 10].groupby('respondent_type').size()
summary['sesiones_completas'] = completed.reindex(summary.index, fill_value=0)

print('Resumen del corpus de respuestas:')
summary

In [ ]:
# Distribución de número de respuestas por sesión
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=session_counts, x='n', hue='respondent_type',
    bins=range(1, 12), multiple='stack', ax=ax,
)
ax.set(
    title='Rondas completadas por sesión',
    xlabel='Número de rondas',
    ylabel='Sesiones',
)
ax.axvline(10, color='red', linestyle='--', linewidth=1, alpha=0.6)
ax.text(10.05, ax.get_ylim()[1] * 0.9, 'sesión completa', color='red', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig01_rondas_por_sesion.png')
plt.show()

### 2.2 Tiempo de respuesta

Tiempos de respuesta atípicos pueden indicar respondientes distraídos o sesiones interrumpidas. Se reporta la mediana porque la distribución es marcadamente asimétrica.

In [ ]:
rt = responses.copy()
rt['response_time_s'] = rt['response_time_ms'] / 1000.0

rt_summary = rt.groupby('respondent_type')['response_time_s'].describe(percentiles=[.25, .5, .75]).round(2)
print('Tiempo de respuesta (segundos):')
rt_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxenplot(
    data=rt[rt['response_time_s'] < 90],
    x='respondent_type', y='response_time_s', ax=ax,
)
ax.set(
    title='Distribución del tiempo de respuesta por tipo de respondiente',
    xlabel='Tipo de respondiente',
    ylabel='Tiempo de respuesta (s)',
)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig02_tiempos_respuesta.png')
plt.show()

## 3. Análisis de votación por triplete

Cada triplete recibió decenas de votos. La distribución de votos A vs B revela qué tripletes generan consenso fuerte y cuáles son ambiguos (cercanos a 50/50). Los tripletes ambiguos son los más informativos para distinguir entre modelos, mientras que los unánimes representan casos donde la similitud perceptual es evidente para los humanos.

In [ ]:
vote_counts = (
    responses
    .groupby(['triplet_id', 'choice'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'a': 'votos_A', 'b': 'votos_B'})
)
vote_counts['total']     = vote_counts.sum(axis=1)
vote_counts['pct_A']     = (vote_counts['votos_A'] / vote_counts['total'] * 100).round(1)
vote_counts['mayoria']   = np.where(vote_counts['votos_A'] >= vote_counts['votos_B'], 'a', 'b')
vote_counts['fuerza']    = (np.maximum(vote_counts['votos_A'], vote_counts['votos_B']) / vote_counts['total']).round(2)
vote_counts = vote_counts.merge(triplets[['id','anchor_filename','option_a_filename','option_b_filename']], left_index=True, right_on='id').set_index('id')

print('Distribución de votos por triplete (ordenado por ambigüedad):')
vote_counts.sort_values('fuerza').round(2)

In [ ]:
def compute_vote_counts(resp_subset):
    vc = (
        resp_subset
        .groupby(['triplet_id', 'choice'])
        .size()
        .unstack(fill_value=0)
        .rename(columns={'a': 'votos_A', 'b': 'votos_B'})
    )
    for col in ['votos_A', 'votos_B']:
        if col not in vc.columns:
            vc[col] = 0
    vc['total'] = vc['votos_A'] + vc['votos_B']
    vc['pct_A'] = (vc['votos_A'] / vc['total'] * 100).round(1)
    return vc

vc_public = compute_vote_counts(responses[responses['respondent_type'] == 'public'])
vc_expert = compute_vote_counts(responses[responses['respondent_type'] == 'expert'])
vc_all    = compute_vote_counts(responses)

sort_order = vc_all.sort_values('pct_A').index
col_A, col_B = sns.color_palette('deep')[0], sns.color_palette('deep')[3]

groups = [
    ('fig03a_votos_publico.png',  'Distribución de votos por triplete — Público general', vc_public),
    ('fig03b_votos_expertos.png', 'Distribución de votos por triplete — Expertos',        vc_expert),
    ('fig03c_votos_global.png',   'Distribución de votos por triplete — Global (ambos)',  vc_all),
]

for fname, title, vc in groups:
    vc_plot = vc.reindex(sort_order).fillna(0)
    y_pos = np.arange(len(vc_plot))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(y_pos, vc_plot['votos_A'], color=col_A, label='Voto A')
    ax.barh(y_pos, vc_plot['votos_B'], left=vc_plot['votos_A'], color=col_B, label='Voto B')
    for i, (idx, row) in enumerate(vc_plot.iterrows()):
        if row['total'] > 0:
            ax.text(row['total'] + 0.5, i, f"{row['pct_A']:.0f}% A", va='center', fontsize=9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'T{idx}' for idx in vc_plot.index])
    ax.axvline(vc_plot['total'].max() / 2, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set(title=title, xlabel='Número de votos', ylabel='Triplete')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname)
    plt.show()

## 4. Concordancia entre expertos y público

¿Los expertos en audio votan de manera distinta al público general? Esta sección compara el voto mayoritario del subgrupo *expert* contra el del subgrupo *public* en cada triplete, usando dos métricas:

- **Acuerdo simple**: porcentaje de tripletes donde ambos grupos eligen la misma opción.
- **Cohen's κ**: corregido por concordancia esperada al azar (κ=1 acuerdo perfecto, κ=0 acuerdo equivalente al azar).

In [ ]:
def majority_choice(group):
    """Devuelve la opción mayoritaria; los empates se consideran indeterminados."""
    counts = Counter(group)
    if not counts:
        return None
    ranked = counts.most_common()
    if len(ranked) > 1 and ranked[0][1] == ranked[1][1]:
        return None
    return ranked[0][0]

majority_by_group = (
    responses
    .groupby(['triplet_id', 'respondent_type'])['choice']
    .apply(majority_choice)
    .unstack()
)
has_both = majority_by_group[['expert', 'public']].notna().all(axis=1)
majority_by_group['acuerdo'] = pd.NA
majority_by_group.loc[has_both, 'acuerdo'] = (
    majority_by_group.loc[has_both, 'expert'] == majority_by_group.loc[has_both, 'public']
)
print('Voto mayoritario por triplete y grupo:')
majority_by_group

In [ ]:
valid = majority_by_group.dropna(subset=['expert', 'public'])
if len(valid) >= 2:
    agreement = valid['acuerdo'].mean()
    kappa = cohen_kappa_score(valid['expert'], valid['public'])
    print(f'Tripletes con voto válido en ambos grupos: {len(valid)}/{len(majority_by_group)}')
    print(f'Acuerdo simple: {agreement*100:.1f}%')
    print(f"Cohen's κ:      {kappa:.3f}")
else:
    print('No hay suficientes tripletes con votos en ambos grupos para calcular acuerdo.')

## 5. Cómputo del Human-Model Agreement (HMA)

Para cada triplete y cada modelo:
1. Se extraen los embeddings del *anchor* y de las opciones A y B.
2. Se calcula la distancia coseno `d(anchor, A)` y `d(anchor, B)`.
3. La predicción del modelo es **A si `d_A ≤ d_B`**, **B en caso contrario**.
4. Se compara la predicción contra el voto mayoritario humano.

El HMA del modelo es el porcentaje de tripletes donde modelo y humanos coinciden.

In [ ]:
def blob_to_numpy(blob):
    return np.load(io.BytesIO(blob), allow_pickle=False)

def load_embeddings(model, dataset, filenames, conn):
    placeholder = ','.join('?' * len(filenames))
    query = f'''
        SELECT t.filename, e.embedding_data
        FROM embeddings e
        JOIN tracks t ON e.track_id = t.id
        WHERE t.filename IN ({placeholder})
          AND e.model = ? AND e.dataset = ?
    '''
    rows = conn.execute(query, (*filenames, model, dataset)).fetchall()
    return {fname: blob_to_numpy(blob).reshape(1, -1) for fname, blob in rows}

all_filenames = sorted(set(
    triplets['anchor_filename'].tolist()
    + triplets['option_a_filename'].tolist()
    + triplets['option_b_filename'].tolist()
))
print(f'Filenames únicos en tripletes: {len(all_filenames)}')

In [ ]:
# Voto mayoritario humano (global) por triplete
human_majority = responses.groupby('triplet_id')['choice'].apply(majority_choice)

# Subset por grupo (para HMA condicional)
expert_majority = responses[responses['respondent_type']=='expert'].groupby('triplet_id')['choice'].apply(majority_choice)
public_majority = responses[responses['respondent_type']=='public'].groupby('triplet_id')['choice'].apply(majority_choice)

# Calcular predicciones de cada modelo
model_predictions = {}  # {model_key: {triplet_id: 'a'|'b'}}
missing_log = []

with sqlite3.connect(EMBED_DB) as econn:
    for model_key, dataset, _ in MODELS:
        emb_map = load_embeddings(model_key, dataset, all_filenames, econn)
        preds = {}
        for _, t in triplets.iterrows():
            anc = emb_map.get(t['anchor_filename'])
            ea  = emb_map.get(t['option_a_filename'])
            eb  = emb_map.get(t['option_b_filename'])
            if anc is None or ea is None or eb is None:
                missing_log.append((model_key, t['id']))
                continue
            da = float(cosine_distances(anc, ea)[0,0])
            db = float(cosine_distances(anc, eb)[0,0])
            preds[t['id']] = 'a' if da <= db else 'b'
        model_predictions[model_key] = preds
        print(f'{model_key}: {len(preds)}/{len(triplets)} tripletes con predicción')

if missing_log:
    print('\nFaltantes:', missing_log)

In [ ]:
def compute_hma(predictions, human_choices):
    """Devuelve (k, n, hma_pct, ci_low_pct, ci_high_pct)."""
    human_choices = human_choices.dropna()
    common = set(predictions) & set(human_choices.index)
    if not common:
        return 0, 0, np.nan, np.nan, np.nan
    k = sum(predictions[t] == human_choices[t] for t in common)
    n = len(common)
    hma = k / n if n else np.nan
    ci_low, ci_high = proportion_confint(k, n, alpha=0.05, method='wilson')
    return k, n, hma*100, ci_low*100, ci_high*100

rows = []
for model_key, dataset, display in MODELS:
    preds = model_predictions[model_key]
    k_all,    n_all,    h_all,    lo_all,    hi_all    = compute_hma(preds, human_majority)
    k_exp,    n_exp,    h_exp,    lo_exp,    hi_exp    = compute_hma(preds, expert_majority)
    k_pub,    n_pub,    h_pub,    lo_pub,    hi_pub    = compute_hma(preds, public_majority)
    rows.append({
        'modelo': display,
        'model_key': model_key,
        'HMA_global': h_all, 'IC95_lo': lo_all, 'IC95_hi': hi_all, 'k_n': f'{k_all}/{n_all}',
        'HMA_expert': h_exp, 'k_n_expert': f'{k_exp}/{n_exp}',
        'HMA_public': h_pub, 'k_n_public': f'{k_pub}/{n_pub}',
    })

hma_df = pd.DataFrame(rows).round(1)
print('Human-Model Agreement por modelo (%):')
hma_df

In [ ]:
# Bar chart con error bars (Wilson CI 95%)
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(hma_df))
vals  = hma_df['HMA_global'].values
errs  = np.array([
    vals - hma_df['IC95_lo'].values,
    hma_df['IC95_hi'].values - vals,
])
bars = ax.bar(x_pos, vals, yerr=errs, capsize=8, color=sns.color_palette('deep'))
ax.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.text(len(hma_df)-0.4, 51, 'azar (50%)', color='gray', fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(hma_df['modelo'], rotation=15, ha='right')
ax.set_ylabel('HMA (%)')
ax.set_ylim(0, 105)
ax.set_title('Human-Model Agreement por modelo\n(IC al 95% Wilson)')
for x, v in zip(x_pos, vals):
    ax.text(x, v + 2, f'{v:.0f}%', ha='center', fontsize=10, fontweight='bold')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig04_hma_global.png')
plt.show()

In [ ]:
# HMA: expertos vs público — bar chart agrupado
long = pd.melt(
    hma_df,
    id_vars=['modelo'],
    value_vars=['HMA_expert', 'HMA_public', 'HMA_global'],
    var_name='grupo', value_name='HMA',
)
long['grupo'] = long['grupo'].map({
    'HMA_expert': 'Expertos',
    'HMA_public': 'Público',
    'HMA_global': 'Global',
})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=long, x='modelo', y='HMA', hue='grupo', ax=ax)
ax.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()], rotation=15, ha='right')
ax.set_ylabel('HMA (%)')
ax.set_ylim(0, 105)
ax.set_title('Human-Model Agreement: expertos vs público vs global')
ax.legend(title='', loc='lower right')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig05_hma_por_grupo.png')
plt.show()

## 6. Heatmap por triplete × modelo

Muestra qué tripletes son fáciles (todos los modelos coinciden con humanos) y cuáles son difíciles (modelos discrepan). Útil para identificar regiones del espacio perceptual donde unos modelos fallan sistemáticamente.

In [ ]:
matrix = pd.DataFrame(index=sorted(triplets['id']), columns=[m for m,_,_ in MODELS], dtype=float)
for model_key, _, _ in MODELS:
    preds = model_predictions[model_key]
    for tid in matrix.index:
        if tid in preds and tid in human_majority.index:
            matrix.loc[tid, model_key] = 1 if preds[tid] == human_majority[tid] else 0

matrix_display = matrix.copy()
matrix_display.columns = [MODEL_NAMES[c] for c in matrix.columns]
matrix_display.index = [f'T{i}' for i in matrix.index]

fig, ax = plt.subplots(figsize=(8, max(4, 0.4*len(matrix))))
sns.heatmap(
    matrix_display.astype(float),
    annot=True, fmt='.0f',
    cmap=sns.color_palette(['#c0392b', '#27ae60'], as_cmap=False),
    cbar=False, linewidths=0.5, linecolor='white',
    ax=ax,
)
ax.set_title('Concordancia por triplete y modelo\n(verde = coincide con humanos, rojo = no coincide)')
ax.set_xlabel('')
ax.set_ylabel('Triplete')
plt.xticks(rotation=15, ha='right')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig06_heatmap_triplete_modelo.png')
plt.show()

## 7. Comparación pareada entre modelos (test de McNemar)

El test de McNemar es el test apropiado para comparar dos clasificadores sobre el mismo conjunto de pruebas. Evalúa si los desacuerdos entre dos modelos están sesgados a favor de uno u otro. Reportamos p-valor; valores < 0.05 indican diferencia significativa.

In [ ]:
from itertools import combinations

pair_rows = []
model_keys = [m for m,_,_ in MODELS]
for m1, m2 in combinations(model_keys, 2):
    common = sorted(set(model_predictions[m1]) & set(model_predictions[m2]) & set(human_majority.index))
    a = sum(1 for t in common if model_predictions[m1][t]==human_majority[t] and model_predictions[m2][t]!=human_majority[t])
    b = sum(1 for t in common if model_predictions[m1][t]!=human_majority[t] and model_predictions[m2][t]==human_majority[t])
    table = [[0, a],[b, 0]]
    if (a + b) >= 1:
        result = mcnemar(table, exact=True)
        pval = result.pvalue
    else:
        pval = np.nan
    pair_rows.append({
        'modelo_A': MODEL_NAMES[m1],
        'modelo_B': MODEL_NAMES[m2],
        'A_acierta_y_B_no': a,
        'B_acierta_y_A_no': b,
        'p_valor': pval,
    })

mcnemar_df = pd.DataFrame(pair_rows).round(4)
print('McNemar pareado (n=10 tripletes; con tan pocos datos los p-valores son orientativos):')
mcnemar_df

## 8. Exportación para la tesis

Todas las tablas se guardan como CSV en `backend/reports/tables/` y todas las figuras como PNG en `backend/reports/figures/`.

In [ ]:
summary.to_csv(TBL_DIR / 'tabla01_resumen_respondientes.csv')
rt_summary.to_csv(TBL_DIR / 'tabla02_tiempos_respuesta.csv')
vote_counts.to_csv(TBL_DIR / 'tabla03_votos_por_triplete.csv')
majority_by_group.to_csv(TBL_DIR / 'tabla04_mayoria_por_grupo.csv')
hma_df.to_csv(TBL_DIR / 'tabla05_hma_por_modelo.csv', index=False)
mcnemar_df.to_csv(TBL_DIR / 'tabla06_mcnemar_pareado.csv', index=False)
matrix_display.to_csv(TBL_DIR / 'tabla07_concordancia_triplete_modelo.csv')

print('Archivos generados en', TBL_DIR)
for p in sorted(TBL_DIR.glob('*.csv')):
    print(' ', p.name)
print('\nFiguras generadas en', FIG_DIR)
for p in sorted(FIG_DIR.glob('*.png')):
    print(' ', p.name)

## 9. Resumen para la tesis

Las celdas siguientes producen un texto consolidado con los hallazgos principales, listo para citarse en la sección de resultados.

In [ ]:
best = hma_df.sort_values('HMA_global', ascending=False).iloc[0]
worst = hma_df.sort_values('HMA_global', ascending=False).iloc[-1]
n_tripletes = len(triplets)
n_resp = len(responses)
n_sesiones = responses['session_id'].nunique()
n_completas = (session_counts['n']>=10).sum()
n_expertos = responses[responses['respondent_type']=='expert']['session_id'].nunique()
n_publico  = responses[responses['respondent_type']=='public']['session_id'].nunique()

print(f'''
RESUMEN EJECUTIVO
=================

Corpus de respuestas:
  - {n_tripletes} tripletes evaluados
  - {n_resp} respuestas individuales
  - {n_sesiones} sesiones únicas ({n_completas} completas con ≥10 rondas)
  - {n_publico} respondientes del público + {n_expertos} expertos

Concordancia humano-modelo:
  - Mejor modelo:  {best["modelo"]}  ({best["HMA_global"]:.0f}% HMA, IC95% [{best["IC95_lo"]:.0f}, {best["IC95_hi"]:.0f}])
  - Peor modelo:   {worst["modelo"]}  ({worst["HMA_global"]:.0f}% HMA, IC95% [{worst["IC95_lo"]:.0f}, {worst["IC95_hi"]:.0f}])
  - Diferencia:    {best["HMA_global"]-worst["HMA_global"]:.0f} puntos porcentuales

Caveats:
  - Con {n_tripletes} tripletes los IC son amplios; los resultados son exploratorios.
  - Para detectar diferencias de ~10% entre modelos con potencia suficiente se recomienda
    expandir a 30–50 tripletes en futuras iteraciones.
''')